<div style="background:linear-gradient(135deg,#117A65 0%,#1E8449 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#A9DFBF;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 3 · CLASE 9</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Interpretación, Clasificación y Casos Aplicados</h1>
  <p style="color:#A9DFBF;font-size:16px;margin:0 0 24px 0;font-style:italic;">Scores discriminantes · QDA · Error por clase · Caso marketing</p>
  <hr style="border-color:#52BE80;margin:0 0 20px 0;">
  <p style="color:#EAFAF1;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; <strong>Módulo 3:</strong> Análisis Discriminante &nbsp;·&nbsp; 2 horas</p>
</div>

## Objetivos
| # | Al terminar podés |
|---|-------------------|
| 1 | Calcular e interpretar scores discriminantes |
| 2 | Implementar QDA y saber cuándo supera a LDA |
| 3 | Analizar la matriz de confusión por clase |
| 4 | Construir un pipeline completo de segmentación de clientes |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('Setup listo')

---
## 1. Scores discriminantes e interpretación

El **score discriminante** de una observación es su proyección sobre el eje LD:

$$\text{score}_i = \mathbf{w}^\top \mathbf{x}_i$$

Más alejado del centro → más confianza en la clasificación.

In [ ]:
# Dataset: 3 segmentos de clientes
np.random.seed(SEED)
n_seg=150
Sigma_seg=np.array([[2.,0.6,0.2],[0.6,2.,0.4],[0.2,0.4,1.]])
mus_seg=[np.array([-2.5,-2,1]),np.array([0,1,-1]),np.array([2.5,2,0])]
X_seg=np.vstack([np.random.multivariate_normal(mu,Sigma_seg,n_seg) for mu in mus_seg])
y_seg=np.repeat(['Bajo','Medio','Alto'],n_seg)
Xtr_s,Xte_s,ytr_s,yte_s=train_test_split(X_seg,y_seg,test_size=0.25,random_state=SEED)
lda_s=LinearDiscriminantAnalysis(); lda_s.fit(Xtr_s,ytr_s)
scores_s=lda_s.transform(Xte_s)   # proyección en espacio LD

fig,axes=plt.subplots(1,2,figsize=(13,5))
colores_s={'Bajo':'#E74C3C','Medio':'#F39C12','Alto':'#1E8449'}
for cls,col in colores_s.items():
    m=np.array(yte_s)==cls
    axes[0].scatter(Xte_s[m,0],Xte_s[m,1],s=22,alpha=0.6,color=col,label=cls)
axes[0].set(xlabel='Feature 1',ylabel='Feature 2',title='Espacio original')
axes[0].legend(); axes[0].grid(True,alpha=0.2)
for cls,col in colores_s.items():
    m=np.array(yte_s)==cls
    axes[1].scatter(scores_s[m,0],scores_s[m,1],s=22,alpha=0.7,color=col,label=cls)
acc_s=accuracy_score(yte_s,lda_s.predict(Xte_s))
axes[1].set(xlabel='LD1',ylabel='LD2',title=f'Espacio discriminante (Acc={acc_s:.3f})')
axes[1].legend(); axes[1].grid(True,alpha=0.2)
plt.tight_layout(); plt.show()
print('Score = distancia al centroide en espacio discriminante')
print('Mayor distancia → mayor confianza en la clasificación')

In [ ]:
# Probabilidades vs scores: relación directa
proba_s=lda_s.predict_proba(Xte_s)
# Ordenar por score LD1
orden=np.argsort(scores_s[:,0])
print('Muestra — score LD1, probabilidades predichas y clase real:')
print(f'{"Score LD1":>10s} {"P(Bajo)":>8s} {"P(Medio)":>8s} {"P(Alto)":>8s} {"Clase real":>12s}')
print('─'*52)
for i in orden[:5].tolist()+orden[-5:].tolist():
    print(f'{scores_s[i,0]:>10.3f} {proba_s[i,0]:>8.3f} {proba_s[i,1]:>8.3f} {proba_s[i,2]:>8.3f} {yte_s[i]:>12s}')

---
## 2. QDA — cuando las covarianzas difieren por clase

**LDA:** asume Σ₀ = Σ₁ = ⋯ = Σ (covarianza común)

**QDA:** permite Σₖ distinta por clase → frontera cuadrática

$$\delta_k(x) = -\frac{1}{2}\log|\Sigma_k| - \frac{1}{2}(x-\mu_k)^\top\Sigma_k^{-1}(x-\mu_k) + \log\pi_k$$

In [ ]:
# Escenario: covarianzas MUY distintas — LDA falla, QDA gana
np.random.seed(SEED); n_q=300
X0_q=np.random.multivariate_normal([0,0],[[4,1],[1,1]],n_q)
X1_q=np.random.multivariate_normal([3,3],[[0.5,0],[0,4]],n_q)  # Σ muy distinta
X_q=np.vstack([X0_q,X1_q]); y_q=np.array([0]*n_q+[1]*n_q)
Xtr_q,Xte_q,ytr_q,yte_q=train_test_split(X_q,y_q,test_size=0.25,random_state=SEED)

results_q={}
for nm,m in [('LDA',LinearDiscriminantAnalysis()),
             ('QDA',QuadraticDiscriminantAnalysis()),
             ('Logistica',LogisticRegression(C=1e6,solver='lbfgs',max_iter=500))]:
    m.fit(Xtr_q,ytr_q)
    acc=accuracy_score(yte_q,m.predict(Xte_q))
    auc=roc_auc_score(yte_q,m.predict_proba(Xte_q)[:,1])
    results_q[nm]={'modelo':m,'acc':acc,'auc':auc}
    print(f'  {nm:12s}: Acc={acc:.4f}  AUC={auc:.4f}')

print('\nCovarianzas reales por clase:')
print(f'  Clase 0: diag={np.diag(np.cov(X0_q.T)).round(2)}')
print(f'  Clase 1: diag={np.diag(np.cov(X1_q.T)).round(2)}')
print('→ QDA debería superar a LDA por las covarianzas muy distintas')

In [ ]:
# Visualizar fronteras de decisión LDA vs QDA
xx,yy=np.meshgrid(np.linspace(-5,9,200),np.linspace(-5,10,200))
XX=np.c_[xx.ravel(),yy.ravel()]
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,(nm,d) in zip(axes,[('LDA',results_q['LDA']),('QDA',results_q['QDA'])]):
    Z=d['modelo'].predict(XX).astype(float); Z=Z.reshape(xx.shape)
    ax.contourf(xx,yy,Z,alpha=0.15,cmap='RdYlGn',levels=[-0.5,0.5,1.5])
    ax.contour(xx,yy,Z,colors='gray',linewidths=1,levels=[0.5])
    ax.scatter(X0_q[:,0],X0_q[:,1],s=12,alpha=0.4,color='#E74C3C',label='Clase 0')
    ax.scatter(X1_q[:,0],X1_q[:,1],s=12,alpha=0.4,color='#1E8449',label='Clase 1')
    ax.set(title=f'{nm} — Acc={d["acc"]:.3f}  AUC={d["auc"]:.3f}')
    ax.legend(fontsize=9); ax.grid(True,alpha=0.15)
plt.suptitle('Frontera LDA (lineal) vs QDA (cuadrática)',fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3. Matriz de confusión y error por clase

In [ ]:
# Análisis detallado de errores por clase
np.random.seed(SEED); n_err=200
Sigma_err=np.array([[2,0.5,0],[0.5,2,0.3],[0,0.3,1]])
X_err=np.vstack([np.random.multivariate_normal(mu,Sigma_err,n_err) for mu in [[-2,-2,0],[0,1.5,-1],[2.5,-1,1]]])
y_err=np.repeat(['Segmento_A','Segmento_B','Segmento_C'],n_err)
Xtr_e,Xte_e,ytr_e,yte_e=train_test_split(X_err,y_err,test_size=0.25,random_state=SEED)
lda_e=LinearDiscriminantAnalysis(); lda_e.fit(Xtr_e,ytr_e)
y_pred_e=lda_e.predict(Xte_e)

# Matriz de confusión
cm=confusion_matrix(yte_e,y_pred_e,labels=['Segmento_A','Segmento_B','Segmento_C'])
df_cm=pd.DataFrame(cm,
    index=['Real A','Real B','Real C'],
    columns=['Pred A','Pred B','Pred C'])

print(f'Accuracy: {accuracy_score(yte_e,y_pred_e):.4f}')
print('\nMatriz de confusión:')
print(df_cm)

fig,ax=plt.subplots(figsize=(6,5))
im=ax.imshow(cm,cmap='Greens')
plt.colorbar(im,ax=ax)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(['Pred A','Pred B','Pred C'])
ax.set_yticklabels(['Real A','Real B','Real C'])
ax.set_title('Matriz de confusión — LDA',fontweight='bold')
for i in range(3):
    for j in range(3):
        ax.text(j,i,cm[i,j],ha='center',va='center',fontsize=14,
                color='white' if cm[i,j]>cm.max()*0.55 else 'black')
plt.tight_layout(); plt.show()

# Error por clase
print('\nError de clasificación por clase:')
for cls in ['Segmento_A','Segmento_B','Segmento_C']:
    mask=np.array(yte_e)==cls
    err=1-accuracy_score(np.array(yte_e)[mask],np.array(y_pred_e)[mask])
    print(f'  {cls}: {err:.2%} de error')

---
## 4. Caso completo: segmentación de clientes de marketing

Pipeline real de segmentación con LDA y QDA para una empresa de retail.

In [ ]:
# Dataset de marketing — 3 perfiles de consumidor
np.random.seed(SEED)
n_mkt=800
Sigma_mkt=np.array([[2,0.8,-0.3,0.4],[0.8,3,0.5,0],[-0.3,0.5,2,0.3],[0.4,0,0.3,1.5]])
mus_mkt=[np.array([-2,1,0.5,-1]),    # Familiar: baja frecuencia, alto ticket, pocas online
         np.array([0,-1.5,-0.5,0.5]),# Joven urbano: frecuencia media, online alto
         np.array([2,1,-1,-0.5])]    # Senior: alta frecuencia, bajo ticket, pocas online
X_mkt=np.vstack([np.random.multivariate_normal(mu,Sigma_mkt,n_mkt//3) for mu in mus_mkt])
y_mkt=np.repeat(['Familiar','Joven_Urbano','Senior'],n_mkt//3)
feats_mkt=['freq_visitas','ticket_promedio','compras_online','ratio_descuento']

df_mkt=pd.DataFrame(X_mkt,columns=feats_mkt)
df_mkt['segmento']=y_mkt
print('Perfil promedio por segmento:')
print(df_mkt.groupby('segmento')[feats_mkt].mean().round(2).to_string())

In [ ]:
# Pipeline completo: estandarizar + LDA + QDA + comparar
sc_mkt=StandardScaler(); X_mkt_sc=sc_mkt.fit_transform(X_mkt)
Xtr_mkt,Xte_mkt,ytr_mkt,yte_mkt=train_test_split(X_mkt_sc,y_mkt,test_size=0.2,random_state=SEED)

print('Comparación LDA vs QDA vs Logística:')
print(f'{"Modelo":15s} {"Accuracy":>10s}')
print('─'*28)
best_acc=0; best_model=None; best_nm=''
for nm,m in [('LDA',LinearDiscriminantAnalysis()),
             ('QDA',QuadraticDiscriminantAnalysis()),
             ('Logistica',LogisticRegression(C=1e6,solver='lbfgs',max_iter=1000))]:
    m.fit(Xtr_mkt,ytr_mkt)
    acc=accuracy_score(yte_mkt,m.predict(Xte_mkt))
    print(f'  {nm:13s} {acc:>10.4f}')
    if acc>best_acc: best_acc=acc; best_model=m; best_nm=nm
print(f'\nMejor modelo: {best_nm} (Acc={best_acc:.4f})')

In [ ]:
# Coeficientes discriminantes — importancia de cada feature
if hasattr(best_model,'scalings_'):
    print('Coeficientes LD1 y LD2 (importancia de features):')
    for nm,c1,c2 in zip(feats_mkt,best_model.scalings_[:,0],best_model.scalings_[:,1]):
        print(f'  {nm:22s}: LD1={c1:+.4f}  LD2={c2:+.4f}')

print(f'\nReporte detallado ({best_nm}):')
print(classification_report(yte_mkt,best_model.predict(Xte_mkt)))

In [ ]:
# Proyección 2D y visualización
if hasattr(best_model,'transform'):
    X_proj_mkt=best_model.transform(Xte_mkt)
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    colores_mkt={'Familiar':'#1E8449','Joven_Urbano':'#2E86C1','Senior':'#8E44AD'}
    for cls,col in colores_mkt.items():
        m=np.array(yte_mkt)==cls
        axes[0].scatter(Xte_mkt[m,0],Xte_mkt[m,1],s=20,alpha=0.6,color=col,label=cls)
        axes[1].scatter(X_proj_mkt[m,0],X_proj_mkt[m,1],s=20,alpha=0.7,color=col,label=cls)
    axes[0].set(xlabel='freq_visitas (std)',ylabel='ticket_promedio (std)',title='Espacio original')
    axes[0].legend(); axes[0].grid(True,alpha=0.2)
    axes[1].set(xlabel='LD1',ylabel='LD2',title=f'Espacio LDA — separación de segmentos')
    axes[1].legend(); axes[1].grid(True,alpha=0.2)
    plt.suptitle('Segmentación de clientes — Marketing',fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Predecir para 5 nuevos clientes
nuevos_mkt=np.array([
    [0.3, 1.5, -0.8, -0.5],   # cliente A: alta frecuencia
    [-1.5, 0.8, 0.2, -0.8],   # cliente B: pocas visitas, ticket alto
    [0.1, -1.2, 1.5, 0.6],    # cliente C: comprador online
    [1.8, 0.9, -1.1, -0.4],   # cliente D: Senior probablemente
    [-0.2, 0.0, 0.0, 0.1],    # cliente E: perfil neutro
])
nuevos_sc=sc_mkt.transform(nuevos_mkt)
preds_n=best_model.predict(nuevos_sc); probs_n=best_model.predict_proba(nuevos_sc)
print('Predicciones para nuevos clientes:')
print(f'{"Cliente":>9s} {"Segmento":>13s}  {"P(Familiar)":>11s} {"P(Joven_Urb)":>12s} {"P(Senior)":>10s}')
print('─'*60)
for i,(pred,prob) in enumerate(zip(preds_n,probs_n)):
    p_dict=dict(zip(best_model.classes_,prob))
    print(f'  Cliente {chr(65+i)} {pred:>13s}  {p_dict.get("Familiar",0):>11.3f} {p_dict.get("Joven_Urbano",0):>12.3f} {p_dict.get("Senior",0):>10.3f}')

---
## Conclusiones

<div style="background:#EAFAF1;border-left:5px solid #1E8449;padding:20px 24px;border-radius:0 8px 8px 0;">

**01 · Scores = distancias en espacio discriminante**
Mayor distancia del centroide → mayor confianza en la clasificación. Útil para detectar casos inciertos.

**02 · QDA cuando las covarianzas difieren entre clases**
LDA asume Σ común → frontera lineal. QDA estima Σₖ por clase → frontera cuadrática más flexible.

**03 · La matriz de confusión revela qué clases se confunden**
No basta con el accuracy global. Analizar FN/FP por clase para entender dónde falla el modelo.

**04 · En marketing: LDA + proyección = segmentación interpretable**
Los coeficientes LD revelan qué features distinguen cada segmento.

</div>

---
<div style="background:#117A65;color:white;padding:20px 24px;border-radius:8px;">
<strong>Fin Módulo 3 — Análisis Discriminante completado</strong><br>
Próximo: Módulo 4 — PCA · SVD · Varianza explicada · Scree plot<br>
<em>Docente: Josef Rodriguez · Curso 8 · Módulo 3</em>
</div>